# [Video RAG] VectorDB Indexing

In [ ]:
%pip install -Uq langchain langchain-openai langchain-pinecone langchain-text-splitters

In [ ]:
# 구글드라이브 연동
from google.colab import drive
drive.mount('/content/drive')  # 내 구글드라이브를 /content/drive 경로로 마운트

BASE_PATH = '/content/drive/MyDrive/SK네트웍스 Family AI 캠프 23기/05_multimodal_rag'  # 기본작업경로 변수

In [ ]:
import os
from google.colab import userdata  # Colab 비밀키 저장소 접근 모듈

# 환경변수 설정
os.environ['LANGSMITH_TRACING'] = 'true'  # LangSmith 추적 기능 활성화
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = 'skn23-multimodal-rag'
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')

## 비디오별 caption 불러오기
- BASE_PATH/captions/{video_name}_frames.csv

In [ ]:
import os

captions_dir_path = os.path.join(BASE_PATH, 'captions')
csv_files_path = [os.path.join(captions_dir_path, f) for f in os.listdir(captions_dir_path)]
csv_files_path

In [ ]:
import pandas as pd

dfs = []
for f in csv_files_path:
    df = pd.read_csv(f)
    print(f'{os.path.basename(f)}: {df.shape = }')
    dfs.append(df)

# 병합
all_df = pd.concat(dfs, ignore_index=True)
print(all_df.shape)
all_df.head(20)

## chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import uuid

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name='gpt-4.1-mini',
    chunk_size=300, # 토큰수
    chunk_overlap=150,
    separators=['\n\n', '\n', ' ', '']
)

In [ ]:
chunks = []

for idx, row in tqdm(all_df.iterrows(), total=len(all_df)):
    frame_no = row['frame_no']
    video_filename = row['video_filename']
    description = row['description']

    for chunk in text_splitter.split_text(description):
        chunks.append({
            'id': str(uuid.uuid4()),
            'description': chunk,
            'frame_no': frame_no,
            'video_filename': video_filename
        })

chunks_df = pd.DataFrame(chunks)
print(chunks_df.shape)
pd.set_option('display.max_colwidth', None)
chunks_df.head(10)

In [ ]:
# 프레임별 chunk수
chunks_df[['video_filename', 'frame_no']].value_counts().sort_index()

## indexing

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone()
PINECONE_INDEX_NAME = 'video-caption-kr'
index_list = [index.name for index in pc.list_indexes()]

if PINECONE_INDEX_NAME not in index_list:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    print(f'{PINECONE_INDEX_NAME} 인덱스가 생성되었습니다.')
else:
    print(f'{PINECONE_INDEX_NAME} 인덱스가 이미 존재합니다.')

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embedding = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embedding
)

In [ ]:
# add_texts
# add_documents

texts = chunks_df['description'].tolist()
metadatas = [
    {
        'id': row['id'],
        'frame_no': row['frame_no'],
        'video_filename': row['video_filename']
    }
        for idx, row in chunks_df.iterrows()
]

batch_size = 50
for i in tqdm(range(0, len(texts), batch_size)):
    batch_text = texts[i: i + batch_size]
    batch_metadata = metadatas[i: i + batch_size]
    vector_store.add_texts(batch_text, batch_metadata)

In [ ]:
# query = '농구공으로 드리블하는 장면'
# query = '등산하는 장면'
query = '스키를 타고 내려오는 장면'
# query = '축구공으로 드리블하는 장면'
results = vector_store.similarity_search(query, k=3)  # 질문과 가장 유사한 청크 3개 벡터검색

In [ ]:
# from PIL import Image

# 검색결과 metadata를 기반으로 해당 프레임 이미지를 찾아서 화면에 표시하는 함수
def show_image_from_metadata(metadata, frame_dir='frames'):
    video_filename = metadata['video_filename']
    video_basename = os.path.splitext(video_filename)[0]  # 확장자 제거한 파일명으로 사용
    frame_no = int(metadata['frame_no'])
    image_filename = f'{video_basename}_frame{frame_no:05d}.jpg'  # 기존 파일명대로 조립
    image_path = os.path.join(BASE_PATH, frame_dir, video_basename, image_filename)  # frames/{video}/파일명

    image = Image.open(image_path)
    image.thumbnail((480, 270))
    display(image)

show_image_from_metadata(results[0].metadata)

In [ ]:
show_image_from_metadata(results[1].metadata)

In [ ]:
for i, doc in enumerate(results):
    print(f'[{i + 1}]')
    print(doc.page_content)  # 검색된 문서(청크) 본문 텍스트
    print(doc.metadata)      # 문서 메타데이터(파일명/프레임 번호 등)
    show_image_from_metadata(doc.metadata)
    print('\n')